In [1]:
include("src/ContEst.jl")

nonlinear_mpc_θ (generic function with 1 method)

In [2]:
n = 2
m = 2
o = 2

W = Matrix{Float64}(I(n)) * 1.0
V = Matrix{Float64}(I(o)) * 0.1
Q = Matrix{Float64}(I(n)) * 1.0
R = Matrix{Float64}(I(m)) * 0.1
N = 6

function state_dynamics(x, u, θ)
    x⁺ = x .+ u
end

function output_dynamics(x, u, θ)
    y = θ .* x
end

function running_cost(x, u)
    return x' * Q * x + u' * R * u
end

function running_cost_stochastic(ζ, u)
    @assert length(ζ) == n + n * (n + 1) ÷ 2 "Information state has incorrect length."
    x = ζ[1:n] # Extract the state from the information state
    Σ = vec_to_mat(ζ[n+1:end]) # Convert the remaining elements to a covariance matrix
    @assert size(Σ) == (n, n) "Covariance matrix has incorrect size."
    return x' * Q * x + u' * R * u + tr(Q * Σ)
end

function constraints(x, u)
    return -1.0
end


prob = ControlModel(state_dynamics, output_dynamics, (W, V), n, m, o, N, running_cost, running_cost_stochastic, constraints)

ControlModel(Main.state_dynamics, Main.output_dynamics, ([1.0 0.0; 0.0 1.0], [0.1 0.0; 0.0 0.1]), 2, 2, 2, 6, Main.running_cost, Main.running_cost_stochastic, Main.constraints)

In [3]:
Evaluate = nonlinear_mpc_θ(prob)

(::var"#eval#57"{ControlModel, Matrix{VariableRef}, Matrix{VariableRef}, Model, Int64, Int64}) (generic function with 1 method)

In [4]:
# initial conditions and params
x₀₀ = [2.0; 1.0]
Σ₀₀ = W
u₀ = x₀₀
θ = [1.0; 1.0]

2-element Vector{Float64}:
 1.0
 1.0

In [5]:
u₀, J, ∇θJ, feasibility = Evaluate(x₀₀, Σ₀₀, u₀, θ)

([-1.8321595613139081, -0.9160797806569541], 7.4625940263160295, [-0.008693883270944808, -0.008693883270944808], true)

In [6]:
u₀, J1, _, feasibility = Evaluate(x₀₀, Σ₀₀, u₀, θ + [1.0; 0.0] * 1e-6)
u₀, J2, _, feasibility = Evaluate(x₀₀, Σ₀₀, u₀, θ + [-1.0; 0.0] * 1e-6)

([-1.8321595613139074, -0.9160797806569537], 7.4625940350099285, [-0.008693924404967954, -0.008693883270944808], true)

In [7]:
(J1 - J2) / 2e-6

-0.008693881170529494